# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [1]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [2]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    # TODO: implement
    x = x.clone().detach().requires_grad_(True)  # Ensure requires_grad
    f = torch.sum(x**2)
    print(x**2)
    print(f)
    return torch.autograd.grad(outputs=f, inputs=x)[0]
  
x = torch.tensor([1.0,2,3,4])

autograd_res = grad_with_autograd_grad(x)

tensor([ 1.,  4.,  9., 16.], grad_fn=<PowBackward0>)
tensor(30., grad_fn=<SumBackward0>)


In [ ]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    x  = x.clone().detach().requires_grad_(True)

    f = torch.sum(x**2)
    f.backward()
    return x.grad

x = torch.tensor([1.0,2,3,4], requires_grad=True)

grad_res = grad_with_backward(x)

In [ ]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    a  = a.clone().detach().requires_grad_(True)
    b  = b.clone().detach().requires_grad_(True)

    f = torch.sum(a**2+ a*b)
    f.backward()
    return (a.grad, b.grad)

a = torch.arange(5, dtype=torch.float32, requires_grad=True)
b = torch.arange(5,10, dtype=torch.float32,requires_grad=True)

multi_grad_res = grad_wrt_multiple_inputs(a,b)

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [5]:
from torch.utils.data import DataLoader, Dataset

In [6]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return min(len(self.x), len(self.y))

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:

        return (self.x[idx], self.y[idx])
    
x = torch.rand(10)
y = torch.rand(10)

tpds = TensorPairDataset(x,y)
tpds_len = len(tpds) # tpds.__len__()
tpds_getitm_x,tpds_getitm_y  = tpds[2] # tpds.__getitem__(2)

In [7]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        self.input_ids = tokens[:, :-1]
        self.target_ids = tokens[:, 1:]

    def __len__(self) -> int:
        return min(len(self.input_ids), len(self.target_ids))


    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        print(self.input_ids)
        print(self.target_ids)
        return (self.input_ids[idx], self.target_ids[idx])
    
tokens = torch.rand(2, 5)

ntds = NextTokenDataset(tokens)

In [ ]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        self.tokens = tokens
        self.crop_len = crop_len
        self.gen = torch.Generator()
        if (seed != None):
            self.gen.manual_seed(seed)

    def __len__(self) -> int:
        return self.tokens.size(0)

    def __getitem__(self, idx: int) -> torch.Tensor:
        T_total = self.tokens.size(1)
        max_start = T_total - self.crop_len  # s + crop_len <= T_total
        
        s = torch.randint(0, max_start, (1,), generator=self.gen).item()  # scalar
        return self.tokens[idx, s:s+self.crop_len]  

    
tokens = torch.rand(4, 20)
crop_len = 3
seed = 8
rcsds = RandomCropSequenceDataset(tokens, crop_len, seed)

In [9]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    # make sure for any input that the dtype is long (in case test input is not)
    output_tensors =  [seq.long() for seq in seqs]

    # find the max dimension
    max_dim = max(seq.size(0) for seq in output_tensors)
    print("max dim: ", max_dim)

    tokens_tens = torch.empty(len(output_tensors),  max_dim, dtype=torch.long)
    padding_mask_tens = torch.empty(len(output_tensors),  max_dim, dtype=torch.bool)
    lengths_tens = torch.empty(len(output_tensors), dtype=torch.long)


    # for each tensor, pad it and adjust customize the mask (so that non existent values will be False)
    for i, tensor in enumerate(output_tensors):
        tensor_len = tensor.size(0)
        padding_mask_tens[i] = torch.cat((torch.full((1,tensor_len), False), torch.full((1,max_dim-tensor_len), True)), 1)
        pad_tensor = torch.full((max_dim-tensor_len,), pad_value, dtype=torch.long)
        tokens_tens[i] = torch.cat((tensor,pad_tensor), 0)
        lengths_tens[i] = tensor_len
    
    batch = PaddedBatch(tokens=tokens_tens,
                        lengths=lengths_tens,                     
                        padding_mask=padding_mask_tens)
    
    return batch

tensor_list = [torch.tensor([1,2,3,0]),
               torch.tensor([4,5,]),
               torch.tensor([1,2,3,0,5,6]),
               torch.tensor([4,5,6,7,8,9,10])]

pad_res = pad_1d_sequences(tensor_list)

max dim:  7


In [10]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """

    input_ids = [item[0] for item in batch]
    target_ids = [item[1] for item in batch]

    max_len = max(len(input_item) for input_item in input_ids)
    B = len(batch)

    inputs_padded = torch.full((B, max_len), pad_value, dtype=torch.long)
    targets_padded = torch.full((B, max_len), pad_value, dtype=torch.long)
    padding_mask = torch.full((B, max_len), True, dtype=torch.bool)

    for i, (input, target) in enumerate(batch):
        item_length = len(input)
        inputs_padded[i, :item_length] = input
        targets_padded[i, :item_length] = target
        padding_mask[i, :item_length] = False

    attention_mask = ~padding_mask

    batch_dict = {
        'input_ids': inputs_padded,
        'target_ids': targets_padded,
        'attention_mask': attention_mask,    # NOT of padding_mask
        'padding_mask': padding_mask
    }

    return batch_dict

batch = [
    (torch.tensor([101, 25, 42, 7]),   torch.tensor([25, 42, 7, 102])),  # len=4
    (torch.tensor([101, 89]),           torch.tensor([89, 102])),        # len=2
    (torch.tensor([101, 12, 34]),       torch.tensor([12, 34, 102]))     # len=3
]

cntb = collate_next_token_batch(batch)

In [11]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """

    return DataLoader(
    dataset, batch_size, shuffle=shuffle, drop_last=drop_last, 
    collate_fn=collate_fn, num_workers=num_workers
    )




## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [29]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    # initialize under no grad
    with torch.no_grad():
      m = torch.zeros_like(p)
      v = torch.zeros_like(p)
    t = 0

    return AdamWState(m,v,t)




In [ ]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """

    state.t +=1

    m = state.m
    v = state.v
    t = state.t

    m = betas[0]*m + (1-betas[0])*grad
    v = betas[1]*v + (1-betas[1])*grad**2
    m_hat = m/(1-betas[0]**t)
    v_hat = v/(1-betas[1]**t)

    # out of place
    # p = p - lr*(m_hat/(torch.sqrt(v_hat) + eps)+weight_decay*p)
    
    # in place
    p.add_( -lr * (m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p) )

    state.m = m
    state.v = v

    return state

p = torch.rand(3,4)
init_state = init_adamw_state(p)
now_state = adamw_step_(p, p, init_state, 0.1)
print(now_state)

AdamWState(m=tensor([[0.0414, 0.0173, 0.0052, 0.0324],
        [0.0865, 0.0832, 0.0283, 0.0815],
        [0.0908, 0.0226, 0.0627, 0.0190]]), v=tensor([[1.7176e-04, 2.9889e-05, 2.7547e-06, 1.0482e-04],
        [7.4766e-04, 6.9177e-04, 8.0032e-05, 6.6440e-04],
        [8.2441e-04, 5.1052e-05, 3.9375e-04, 3.6031e-05]]), t=1)


In [ ]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """

    assert len(params) == len(grads) == len(states)

    updated_states = []
        
    for p,g,s in zip(params, grads, states):
        updated_states.append(adamw_step_(p=p, grad=g, state=s, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    return updated_states



## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [ ]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    x_batch, y_batch = batch # seperate batch
    model.train() # set model to training mode
    optimizer.zero_grad() # clear prev gradients
    predictions = model(x_batch) # forward pass
    criterion = nn.MSELoss() # define loss function
    loss = criterion(predictions, y_batch) # compute loss
    loss.backward() # backward pass
    optimizer.step() # step optimizer

    return  loss.item()




## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [ ]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""

    return weight.size(1), weight.size(0)


((4, 3),
 tensor([[0.0245, 0.6635, 0.4930, 0.8049],
         [0.4657, 0.0770, 0.4431, 0.2913],
         [0.7071, 0.6657, 0.8916, 0.2216]]))

In [ ]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * torch.sqrt(torch.tensor([6]) / (fan_in + fan_out))
    torch.nn.init.uniform_(weight, -float(bound.tolist()[0]), float(bound.tolist()[0]))  # fill in-place
    return weight

In [ ]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    gain = torch.nn.init.calculate_gain(nonlinearity=nonlinearity)
    std = gain / torch.sqrt(torch.tensor([fan_in]))
    bound = torch.sqrt(torch.tensor([3]))*std
    torch.nn.init.uniform_(weight, -float(bound.tolist()[0]), float(bound.tolist()[0]))  # fill in-place
    return weight

In [ ]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """

    if scheme == "xavier":
        xavier_uniform_(layer.weight)
        torch.nn.init.zeros_(layer.bias)
    if scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight)
        torch.nn.init.zeros_(layer.bias)
    if scheme == "zero":
        torch.nn.init.zeros_(layer.weight)
        torch.nn.init.zeros_(layer.bias)


    return layer

(tensor([3]), float)